# Bronze: Ingest Urban/Rural Population Raw Sources

Default lakehouse: `bronze_lh`. Append-only raw landing — no filtering, no type coercion.
Reads `Files/raw/*.csv` in Fabric, or `./data/*.csv` when run locally.

In [1]:
import os
from datetime import datetime, timezone

try:
    import notebookutils  # noqa: F401  (only importable inside Fabric)
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if not IN_FABRIC:
    from delta import configure_spark_with_delta_pip
    from pyspark.sql import SparkSession
    builder = (
        SparkSession.builder.appName("urbanisation_prospects_bronze")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(builder).getOrCreate()

DEFAULT_LAKEHOUSE = "bronze_lh"
# Paths are relative to this notebook's own directory (Jupyter's default cwd), i.e. repo_root/notebooks/
LOCAL_LAKEHOUSE_ROOT = "../lakehouse"
LOCAL_DATA_ROOT = "../data"


def table_path(lakehouse: str, table: str) -> str:
    if IN_FABRIC:
        mount = "default" if lakehouse == DEFAULT_LAKEHOUSE else lakehouse
        return f"/lakehouse/{mount}/Tables/{table}"
    return f"{LOCAL_LAKEHOUSE_ROOT}/{lakehouse}/Tables/{table}"


def write_delta(lakehouse: str, table: str, df, mode: str = "append"):
    (
        df.write.format("delta")
        .mode(mode)
        .option("mergeSchema", "true")
        # Raw headers contain spaces/parentheses (e.g. "Urban population 1950-2050 (...)");
        # column mapping lets Delta store them verbatim instead of erroring, preserving
        # Bronze's "capture raw input exactly as it arrives" rule.
        .option("delta.columnMapping.mode", "name")
        .save(table_path(lakehouse, table))
    )

In [2]:
from pyspark.sql import functions as F

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

# wid_pretax_income.csv is intentionally excluded — out of scope (see specs/proposal.md)
RAW_SOURCES = [
    {"file": "un_urban_rural_population_1950_2050.csv", "table": "raw_urban_rural_population"},
    {"file": "countries_continents.csv", "table": "raw_countries_continents"},
    {"file": "owid_country_to_who_regions.csv", "table": "raw_who_regions"},
]

In [3]:
for source in RAW_SOURCES:
    raw_path = (
        f"Files/raw/{source['file']}" if IN_FABRIC else f"{LOCAL_DATA_ROOT}/{source['file']}"
    )

    df = (
        spark.read.option("header", True)
        .option("inferSchema", False)
        .csv(raw_path)
        .withColumn("source_file", F.lit(source["file"]))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingested_at", F.current_timestamp())
    )

    write_delta(DEFAULT_LAKEHOUSE, source["table"], df, mode="append")
    print(f"Ingested {df.count()} rows into {source['table']} (batch {BATCH_ID})")

Ingested 27573 rows into raw_urban_rural_population (batch 20260713T211339420261Z)


Ingested 285 rows into raw_countries_continents (batch 20260713T211339420261Z)


Ingested 194 rows into raw_who_regions (batch 20260713T211339420261Z)
